# CI/CDfor AI Apps

**Module 12 · Lesson 12.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why AI CI/CD Is Different
- The Pipeline Is a Graph of Gates
- The Eval Gate: Prompt Regression Under Non-Determinism
- Output Validators: Guard Every Response
- The GitHub Actions Workflow and Caching
- Keyless Deploy with OIDC

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## Every test was green. Quality still dropped.

> 💡 **Info**
>
> You changed one line of a prompt - “be concise” became “be brief and friendly” - and shipped it. Every unit test was green. Then the support tickets spiked: the assistant had started padding every answer with chit-chat, and its accuracy on the hard questions quietly dropped. **Nothing crashed. Nothing threw.** Your tests could not see it, because a traditional test checks that code returns the *right value*, and an LLM app can return a *worse answer* while every assertion still passes. **CI/CD for AI apps is how you catch that automatically.** The same automated pipeline that lints and unit-tests your code also runs an **eval gate** (does this change keep answer quality above the bar?) and **output validators** (is any single response empty, repetitive, or leaking a phone number?), builds the image once, and then deploys it through a **canary** you can roll back in seconds. This lesson builds that whole pipeline as a GitHub Actions workflow, and you can run every piece with no CI runner and no API key.

### 🎯 What you will be able to do after this lesson

- **Build** an AI CI/CD pipeline - lint + tests + an eval gate + output validators + build/scan + a keyless deploy + a canary - as a GitHub Actions workflow plus runnable models of each gate.

- **Compare** traditional deterministic CI with AI CI/CD, and a stored-key deploy with OIDC keyless auth.

- **Operate** an eval gate under non-determinism (pass-rate threshold, repeat + majority-vote judge) and a canary rollout (traffic split + rollback).

- **Evaluate** a pipeline: does it fail fast, build once, gate on quality, deploy without stored keys, and roll back in seconds?

> 📦 **Info**
>
> ✅ Before you startIn **12.5** you built and scanned an immutable, SHA-tagged image - CI/CD is what runs that build automatically and promotes the *same* artifact. In **11.11** you already scored an agent with an LLM-as-judge - here that eval becomes a merge-blocking gate. Building a good eval set is **Lesson 14.4** (Eval-Driven Development); monitoring the deployed app is **Lesson 12.8**; scaling it is **Lesson 12.7**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🚦 **Analogy**
>
> A CI/CD pipeline is an **airport for your code**. Every passenger (a commit) goes through the same fixed sequence of checkpoints in order: check-in, bag drop, security, the gate. Fail any checkpoint and you do not board - you are turned back right there, and the cheap checks come first so you are stopped before you waste time at the far end. An AI app adds two checkpoints a normal traveller skips: a **quality screening** (the eval gate - is this answer still good?) and a **contraband scan on every item** (output validators - no leaked phone numbers). **Where it breaks down:** an airport screens people once; your validators screen every response forever, in production too.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Unit tests are enough for an AI app.”** They are deterministic and stay green even when a prompt edit tanks answer quality. You need an eval gate that judges quality and validators that guard every response.
> - **“An eval must pass exactly, like a unit test.”** LLM-as-judge is non-deterministic - the same response can score differently on two runs. Gate on a **pass-rate threshold** with repeats and majority voting, not on exact agreement.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo deploy habits that will burn you. A **stored service-account JSON key** sitting in your repo secrets - a long-lived credential that leaks, is never rotated, and grants standing access; use **OIDC keyless auth** instead. And **rebuilding a fresh image for each environment** - so production runs an artifact your tests never actually saw. **Build once** and promote the same SHA-tagged image through staging to prod.

---

## 🎯 Concept 1: Why AI CI/CD Is Different

### Why AI CI/CD Is Different

A traditional test is deterministic and checks a value; an LLM app is non-deterministic and can return a worse answer while every unit test stays green. That silent quality regression is what the eval gate catches.

Start with the problem the whole lesson solves. In normal software, a test is a promise: `add(2, 2)` returns `4`, the same way, every time - so an assertion catches any regression. An LLM app breaks both halves of that promise. It is **non-deterministic** (the same prompt can yield different words), and, more dangerously, a change can make the answer *worse* without making it *wrong* in any way a normal assertion can see. Tighten a prompt, add a friendly instruction, swap a model - every unit test stays green while the answers quietly get chattier, less accurate, or off-tone. This is a **silent quality regression**, and it is why an AI pipeline needs a gate that *judges quality*, not just one that checks a return value. The block shows a green unit test hiding a dropped quality score, keyless.

> 📝 **Analogy**
>
> It is the difference between a **spell-checker and an editor**. A spell-checker is deterministic: the word is either in the dictionary or it is not, and it gives the same verdict every time. An editor judges something softer and more important - is the writing *good*? A prompt edit can pass the spell-checker (all the words are real, the JSON is valid, the keyword is present) while an editor would wince at the result. The eval gate is the editor your pipeline was missing.

You change a prompt from “be concise” to “be brief and friendly”. Every unit test still passes. What could a user notice that the tests cannot?

**📝 `01_why_different.py`** - *runnable*

In [ ]:
# A traditional unit test is deterministic; an LLM answer is judged on QUALITY.
def unit_test(fn, x, expected):
    return "PASS" if fn(x) == expected else "FAIL"

def add(x):
    return x + x

# The AI app: a prompt version maps to a scripted answer + a judge quality score (1-10).
ANSWERS = {
    "v1 (be concise)":            {"text": "Take the GenAI Engineering course - it covers agents in depth.", "score": 8.6},
    "v2 (be brief and friendly)": {"text": "Oh hey, great question! You should totally check out our GenAI course, it is amazing!", "score": 5.9},
}
def keyword_test(answer):     # a naive "unit test" on the AI output
    return "PASS" if "GenAI" in answer["text"] else "FAIL"

print("Traditional unit test (deterministic - same result every run):")
for r in range(3):
    print("  run {}: add(2) == 4 -> {}".format(r + 1, unit_test(add, 2, 4)))
print()

print("Prompt edit v1 -> v2 (a 'friendlier' tweak), same keyword unit test:")
for v, a in ANSWERS.items():
    print("  {:<28} keyword_test={}   judge_quality={}/10".format(v, keyword_test(a), a["score"]))
print()
print("The keyword unit test stays GREEN on both versions, but the judge score dropped 8.6 -> 5.9.")
print("A prompt edit silently REGRESSED quality - only an eval gate catches it.")

# Output:
# Traditional unit test (deterministic - same result every run):
#   run 1: add(2) == 4 -> PASS
#   run 2: add(2) == 4 -> PASS
#   run 3: add(2) == 4 -> PASS
#
# Prompt edit v1 -> v2 (a 'friendlier' tweak), same keyword unit test:
#   v1 (be concise)              keyword_test=PASS   judge_quality=8.6/10
#   v2 (be brief and friendly)   keyword_test=PASS   judge_quality=5.9/10
#
# The keyword unit test stays GREEN on both versions, but the judge score dropped 8.6 -> 5.9.
# A prompt edit silently REGRESSED quality - only an eval gate catches it.

- A **unit test** is deterministic - `add(2)` returns the same value on every run, so an assertion is reliable.
- The AI answer is judged on **quality**; the prompt edit v1 → v2 keeps the keyword unit test **GREEN on both**.
- But the judge score **drops from 8.6 to 5.9** - a real regression the keyword test cannot see.
- Only an **eval gate** that scores quality catches this; the rest of the lesson wires that gate into the pipeline.

#### 💡 What just happened

⚡ What just happened?A traditional test checks that code returns the right value; an LLM app is non-deterministic and can return a worse answer while every unit test stays green - a silent quality regression. Tradeoff / the framing for the lesson: an AI pipeline needs two gates a normal one lacks - an eval gate that judges quality and validators that guard every response. Everything else here is how to run those gates automatically.

- Edit a prompt; a deterministic unit-test bar stays green
- An eval-quality gauge drops below the bar - the regression tests cannot see

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Pipeline Is a Graph of Gates

### The Pipeline Is a Graph of Gates

CI/CD is an automated chain of gates triggered on push or pull request: lint, test, eval, build once, scan, deploy, canary. A failed gate stops the line, and the cheapest gates run first.

Zoom out to the shape of the whole thing. **CI** (continuous integration) and **CD** (continuous delivery) are one automated sequence that a code-hosting platform like **GitHub** runs for you on every **push** or **pull request** - no human clicking “deploy”. That sequence is a chain of **gates**: lint → unit tests → the eval gate → build the image → scan it → deploy to staging → smoke test → canary → production. Two rules make it work. First, **a failed gate stops the line** - nothing downstream runs, so a bad change never reaches production. (One setup detail: a red check only truly *blocks a merge* once you mark the job a **required status check** under GitHub branch protection; without that it is advisory and a maintainer could still merge over it.) Second, **put the cheap gates first** so a typo fails in seconds at the lint gate instead of wasting a five-minute build. And crucially, you **build the image once** (tagged with the git SHA, from 12.5) and promote that *same* artifact through every stage. The block runs a commit down the chain, keyless.

> 🚦 **Analogy**
>
> It is the **airport** from the warm-up, made concrete. Check-in, bag drop, security, passport control, the gate - a fixed order, and you clear each before the next. If your bag is overweight at check-in, you are stopped there; you do not get waved through to security and then turned back at the plane. And the quick checks (a boarding pass scan) come before the slow ones (a full bag search), so problems surface early and cheap.

A commit fails the eval gate, which sits before build and deploy. What runs after it?

**📝 `02_pipeline_gates.py`** - *runnable*

In [ ]:
# A CI/CD pipeline is an ordered chain of GATES; a failed gate STOPS the line.
GATES = ["lint", "unit-tests", "eval-gate", "build (once, SHA-tagged)", "scan",
         "deploy-staging", "smoke-test", "canary", "prod"]

def run_pipeline(failing_gate=None):
    image = None
    for g in GATES:
        if failing_gate and g.startswith(failing_gate):
            print("  [FAIL] " + g + "   <- line STOPS here; nothing downstream runs")
            return False
        note = ""
        if g.startswith("build"):
            image = "ai-service:a1b2c3d"; note = "   built " + image
        elif g.startswith("deploy") or g in ("canary", "prod"):
            note = "   deploy " + str(image) + " (same artifact)"
        print("  [PASS] " + g + note)
    return True

print("Commit 1 - every gate passes:")
run_pipeline()
print()
print("Commit 2 - the eval gate fails (a prompt regression):")
run_pipeline(failing_gate="eval-gate")
print()
print("Run the cheapest gates first so a bad change fails fast; one failed gate blocks the deploy.")
print("The image is built ONCE and the SAME artifact flows through staging -> canary -> prod.")

# Output:
# Commit 1 - every gate passes:
#   [PASS] lint
#   [PASS] unit-tests
#   [PASS] eval-gate
#   [PASS] build (once, SHA-tagged)   built ai-service:a1b2c3d
#   [PASS] scan
#   [PASS] deploy-staging   deploy ai-service:a1b2c3d (same artifact)
#   [PASS] smoke-test
#   [PASS] canary   deploy ai-service:a1b2c3d (same artifact)
#   [PASS] prod   deploy ai-service:a1b2c3d (same artifact)
#
# Commit 2 - the eval gate fails (a prompt regression):
#   [PASS] lint
#   [PASS] unit-tests
#   [FAIL] eval-gate   <- line STOPS here; nothing downstream runs
#
# Run the cheapest gates first so a bad change fails fast; one failed gate blocks the deploy.
# The image is built ONCE and the SAME artifact flows through staging -> canary -> prod.

- **Commit 1** passes every gate in order; the image is **built once** and the same `ai-service:a1b2c3d` artifact is deployed to staging, canary, and prod.
- **Commit 2** fails the **eval gate**, and the line **stops there** - build and deploy never run.
- The gates run **cheapest-first** (lint before build) so a bad change fails fast and cheap.
- Build once, promote the same artifact: prod runs exactly what you tested, not a fresh rebuild.

#### 💡 What just happened

⚡ What just happened?A CI/CD pipeline is an ordered chain of gates run automatically on push or pull request; a failed gate stops the line, cheap gates go first, and the image is built once and promoted unchanged. Tradeoff: you invest in the pipeline once, and every change afterward is checked the same way without a human gatekeeper. The AI-specific gates - eval and validators - are the next two steps.

- A commit flows down lint -> unit -> eval -> build -> scan -> deploy -> canary
- Click any gate to fail it; the line stops and nothing downstream runs

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Eval Gate: Prompt Regression Under Non-Determinism

### The Eval Gate: Prompt Regression Under Non-Determinism

A suite of prompt tests scored by an LLM-as-judge; a change must keep the pass rate above a threshold. Because the judge is non-deterministic, you repeat each test and pass by majority, not exact match.

This is the AI-specific gate, and the heart of the lesson. You keep a **suite of prompt tests** - representative inputs with expectations - and score each response with structural checks plus an **LLM-as-judge** (from 11.11). A change is allowed to merge only if it keeps the **pass rate** at or above a threshold; drop below and the gate **blocks the merge**. The wrinkle is non-determinism: the judge might score the same answer 7 on one run and 4 on the next. If you treated it like a unit test - fail on any single low score - a flaky judge would reject good changes constantly. So you **repeat** each test a few times and pass it by **majority** (and run the judge at temperature zero), which tolerates random grader variance while still catching a test that *consistently* fails. Tools like **promptfoo** (a `repeat` / `repeat-min-pass` setting) and **DeepEval** (in pytest) do exactly this. The block runs a suite before and after a prompt edit, keyless.

> 🍴 **Analogy**
>
> It is a **taste test with three judges, not one**. One judge on one day might be in a bad mood and score a good dish low - so you would not fire the chef on a single taste. You have three judges taste it and go with the majority verdict, which cancels out one judge’s random grumpiness but still fails a dish that all three dislike. And you do not need every dish to be perfect - you need the *menu’s pass rate* to stay high. The eval gate is that panel, run automatically on every change.

The judge scores one response 7, 4, 7 across three runs, and your bar is 7. Does this test pass?

**📝 `03_eval_gate.py`** - *runnable*

In [ ]:
# Each prompt test is scored by the judge 3 times (LLM-as-judge is non-deterministic).
# A test passes if a MAJORITY of runs clear the bar; the GATE needs the overall PASS RATE
# to stay at or above a threshold.
JUDGE_BAR = 7      # a single judge run passes at score >= 7
GATE = 0.80        # the suite must keep >= 80% of tests passing

def test_passes(scores):            # majority of 3 runs at or above the bar
    return sum(1 for s in scores if s >= JUDGE_BAR) >= 2

def run_suite(name, suite):
    passed = 0
    print(name + ":")
    for t, scores in suite.items():
        ok = test_passes(scores)
        passed += ok
        print("  {:<16} judge runs {} -> {}".format(t, scores, "PASS" if ok else "FAIL"))
    rate = passed / len(suite)
    verdict = "GATE PASS" if rate >= GATE else "GATE FAIL - block the merge"
    print("  pass rate {}/{} = {:.0%}  (need {:.0%})  -> {}\n".format(passed, len(suite), rate, GATE, verdict))
    return rate

# Baseline on main: 9 of 10 pass. The flaky "summarize" [7,4,7] still passes by majority.
baseline = {
    "greeting": [9, 8, 9], "refund_policy": [8, 8, 7], "safety_refusal": [9, 9, 8],
    "json_output": [7, 8, 8], "summarize": [7, 4, 7], "translate": [8, 7, 8],
    "code_help": [8, 9, 8], "recommend": [7, 7, 8], "tone": [8, 8, 7], "edge_case": [4, 3, 4],
}
run_suite("Baseline (main branch)", baseline)

# After a prompt edit several tests regress below the bar.
edited = dict(baseline)
edited.update({"greeting": [5, 6, 5], "refund_policy": [4, 5, 4], "summarize": [3, 4, 3], "tone": [5, 4, 5]})
run_suite("After the prompt edit (this PR)", edited)

# Output:
# Baseline (main branch):
#   greeting         judge runs [9, 8, 9] -> PASS
#   refund_policy    judge runs [8, 8, 7] -> PASS
#   safety_refusal   judge runs [9, 9, 8] -> PASS
#   json_output      judge runs [7, 8, 8] -> PASS
#   summarize        judge runs [7, 4, 7] -> PASS
#   translate        judge runs [8, 7, 8] -> PASS
#   code_help        judge runs [8, 9, 8] -> PASS
#   recommend        judge runs [7, 7, 8] -> PASS
#   tone             judge runs [8, 8, 7] -> PASS
#   edge_case        judge runs [4, 3, 4] -> FAIL
#   pass rate 9/10 = 90%  (need 80%)  -> GATE PASS
#
# After the prompt edit (this PR):
#   greeting         judge runs [5, 6, 5] -> FAIL
#   refund_policy    judge runs [4, 5, 4] -> FAIL
#   safety_refusal   judge runs [9, 9, 8] -> PASS
#   json_output      judge runs [7, 8, 8] -> PASS
#   summarize        judge runs [3, 4, 3] -> FAIL
#   translate        judge runs [8, 7, 8] -> PASS
#   code_help        judge runs [8, 9, 8] -> PASS
#   recommend        judge runs [7, 7, 8] -> PASS
#   tone             judge runs [5, 4, 5] -> FAIL
#   edge_case        judge runs [4, 3, 4] -> FAIL
#   pass rate 5/10 = 50%  (need 80%)  -> GATE FAIL - block the merge

- Each test is scored by the judge **three times**; it passes if a **majority** of runs clear the bar, so the flaky `summarize` [7, 4, 7] still passes.
- The **baseline** on main passes 9 of 10 - a **90 percent** pass rate, above the 80-percent gate.
- After the prompt edit, four tests regress below the bar, so the pass rate falls to **50 percent**.
- That is below the gate, so it **blocks the merge** - the silent quality regression from Step 1 is caught automatically.

#### 💡 What just happened

⚡ What just happened?The eval gate scores a suite of prompt tests with an LLM-as-judge and requires the pass rate to stay above a threshold; a change that drops it is blocked. Tradeoff / handling non-determinism: repeat each test and pass by majority (temperature zero) so a flaky judge does not reject good changes, while a consistent failure still fails. Building the eval set itself is Lesson 14.4; here you wire it into CI.

- Prompt tests scored by a judge 3 times; toggle repeat/majority
- A pass-rate needle crosses the 80-percent threshold as tests regress

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Output Validators: Guard Every Response

### Output Validators: Guard Every Response

Cheap deterministic checks - empty, repetition, PII leak, format, language - that run in CI on test prompts AND in production on every real response. They catch the obvious failures the expensive eval gate is not run on everything.

The eval gate is powerful but expensive (it calls a judge model), so you cannot run it on every production request. **Output validators** fill that gap: cheap, **deterministic**, keyless checks that run in milliseconds. A handful cover most real failures - the response is **empty** or too short, it is **repetitive** (the model got stuck in a loop), it **leaks PII** (an email or phone number slipped through), it is the wrong **format** (invalid JSON when JSON was promised), or the wrong **language**. The key move is that these run in **two places**: in CI on your test prompts (as another gate), and in **production on every single response** before it reaches the user - because CI only ever sees the prompts you thought to test, while a live request is always something new. The block runs the validators over a few outputs, keyless.

> 👮 **Analogy**
>
> It is a **bouncer at the door**. The bouncer does not judge whether you will have a good time (that is the eval gate’s subtle job) - they do fast, unambiguous checks: are you on the list, are you carrying something you should not be, are you even conscious. Cheap, instant, and applied to *every* guest, not a sample. Output validators are that bouncer, standing at the door of every response your app returns, in test and in production alike.

Your validators pass in CI on all your test prompts. Why still run them in production?

**📝 `04_output_validators.py`** - *runnable*

In [ ]:
import re
# Output validators: cheap DETERMINISTIC guards. Run in CI on test prompts AND in
# production on every response, before it reaches the user.
def validate(output, expected_language="en"):
    issues = []
    if not output or len(output.strip()) < 5:
        issues.append(("critical", "empty or too short"))
    sents = [s.strip() for s in output.split(".") if len(s.strip()) > 10]
    if len(sents) > 4:
        uniq = len(set(s.lower() for s in sents))
        if uniq < len(sents) * 0.5:
            issues.append(("high", "repetitive: {} unique of {} sentences".format(uniq, len(sents))))
    pii = {"email": r"\b[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}\b", "phone": r"\b(?:\+91[\s-]?)?[6-9]\d{9}\b"}
    for kind, pat in pii.items():
        if re.search(pat, output):
            issues.append(("critical", "PII leak: {} pattern".format(kind)))
    if expected_language == "en":
        non_ascii = sum(1 for c in output if ord(c) > 127) / max(len(output), 1)
        if non_ascii > 0.3:
            issues.append(("medium", "expected English but {:.0%} non-ASCII".format(non_ascii)))
    critical = any(sev == "critical" for sev, _ in issues)
    return issues, critical

OUTPUTS = [
    "MCP is an open standard by Anthropic for connecting AI models to tools and data.",
    "",
    "The answer is 42. The answer is 42. The answer is 42. The answer is 42. The answer is 42.",
    "Sure, reach our team at admin@netsetos.com or call +91 9876543210 any time.",
]
print("Output validators - deterministic, keyless, run in CI and in production:")
for o in OUTPUTS:
    issues, critical = validate(o)
    label = o[:44] + ("..." if len(o) > 44 else "")
    verdict = "PASS " if not issues else ("BLOCK" if critical else "WARN ")
    print('  [{}] "{}"'.format(verdict, label))
    for sev, msg in issues:
        print("          - [{}] {}".format(sev, msg))

# Output:
# Output validators - deterministic, keyless, run in CI and in production:
#   [PASS ] "MCP is an open standard by Anthropic for con..."
#   [BLOCK] ""
#           - [critical] empty or too short
#   [WARN ] "The answer is 42. The answer is 42. The answ..."
#           - [high] repetitive: 1 unique of 5 sentences
#   [BLOCK] "Sure, reach our team at admin@netsetos.com o..."
#           - [critical] PII leak: email pattern
#           - [critical] PII leak: phone pattern

- Each response runs through five **deterministic** checks; a clean answer **passes**.
- An **empty** response and one leaking an **email and phone number** are **BLOCKED** (critical).
- A **repetitive** response (one unique sentence of five) is a non-critical **WARN**.
- These are fast and keyless, so they run in CI as a gate **and** in production on every response - the eval gate is too costly to run on all of them.

#### 💡 What just happened

⚡ What just happened?Output validators are cheap deterministic guards - empty, repetition, PII, format, language - that run in CI on test prompts and in production on every response. Tradeoff: they catch obvious, unambiguous failures fast, but not subtle quality (that is the eval gate’s job). Run both: the eval gate on a test suite in CI, the validators everywhere including live traffic, because production always sees inputs CI never tested.

- Four sample outputs fall through the five deterministic validators
- Each lights pass / warn / block with a severity chip

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: The GitHub Actions Workflow and Caching

### The GitHub Actions Workflow and Caching

A workflow is YAML that runs jobs of steps on every push or PR. Caching dependencies and Docker layers is the single biggest speedup, and a matrix runs configurations in parallel.

Now the concrete tool. On **GitHub**, a pipeline is a **workflow** - a YAML file in `.github/workflows/` that declares **when** it runs (`on: [push, pull_request]`) and **what** it does (`jobs`, each a list of `steps`) on a fresh runner. The first thing you notice in a real workflow is **caching**, because it is the single biggest speedup: dependencies and Docker image layers that did not change are **reused** instead of downloaded and rebuilt every run (GitHub gives each repo a cache). Set `enable-cache` on the uv setup step and use the Docker layer cache, and a cold ninety-second install becomes a warm few seconds. The second thing is a **matrix**: declare a list (say Python 3.12 and 3.13) and the job runs once per entry **in parallel**, so testing two configurations costs the wall-clock of the slower one, not the sum. The block models a run cold versus cached and a parallel matrix, keyless.

> 🍳 **Analogy**
>
> A workflow is a **recipe card the kitchen runs on every order** - the same steps, in the same order, no chef improvising. Caching is the **stocked pantry**: you do not run to the store for flour on every single order, you keep it on the shelf and grab it in seconds. And a matrix is **two identical stations cooking the dish two ways at once** - one gas, one induction - so you learn how both turn out in the time it takes to cook one, not two.

Your CI run is slow. Which change cuts the most time for a typical AI app?

**📝 `05_workflow_caching.py`** - *runnable*

In [ ]:
# A GitHub Actions run is a list of steps, each with a cold and a cached duration (seconds).
STEPS = [
    ("checkout",        3,   3),
    ("setup deps (uv)", 48,  4),    # the big win: cached dependencies
    ("lint",            6,   6),
    ("unit tests",      22,  22),
    ("eval gate",       35,  35),
    ("docker build",    70,  9),    # cached layers
    ("scan (trivy)",    12,  12),
]
cold = sum(c for _, c, _ in STEPS)
warm = sum(w for _, _, w in STEPS)
print("A workflow run - cold cache vs warm cache:")
print("  {:<18}{:>7}{:>9}".format("step", "cold", "cached"))
for name, c, w in STEPS:
    print("  {:<18}{:>6}s{:>8}s".format(name, c, w))
print("  {:<18}{:>6}s{:>8}s   -> {:.0%} faster with the cache".format("TOTAL", cold, warm, 1 - warm / cold))
print()

# A matrix runs configs in PARALLEL, so wall-clock is the slowest lane, not the sum.
MATRIX = {"py3.12": warm, "py3.13": warm + 3}
serial = sum(MATRIX.values())
parallel = max(MATRIX.values())
print("A matrix (Python 3.12 + 3.13) runs in parallel:")
for k, v in MATRIX.items():
    print("  {:<8} {}s".format(k, v))
print("  serial would be {}s; parallel wall-clock is {}s (the slowest lane).".format(serial, parallel))

# Output:
# A workflow run - cold cache vs warm cache:
#   step                 cold   cached
#   checkout               3s       3s
#   setup deps (uv)       48s       4s
#   lint                   6s       6s
#   unit tests            22s      22s
#   eval gate             35s      35s
#   docker build          70s       9s
#   scan (trivy)          12s      12s
#   TOTAL                196s      91s   -> 54% faster with the cache
#
# A matrix (Python 3.12 + 3.13) runs in parallel:
#   py3.12   91s
#   py3.13   94s
#   serial would be 185s; parallel wall-clock is 94s (the slowest lane).

- Each step has a **cold** and a **cached** duration; the big wins are **dependency setup** and the **Docker build**, whose layers are reused.
- Caching cuts the total run by **about half** - unchanged deps and layers are not redone.
- A **matrix** of Python 3.12 and 3.13 runs the job **in parallel**.
- So the wall-clock is the **slowest lane**, not the sum of both - you test two configs for roughly the price of one.
- Notice the **eval gate** is the one step caching cannot speed up (35s cold *and* cached) - it calls the LLM judge, so it spends real tokens on every PR, the running cost of gating on quality.

#### 💡 What just happened

⚡ What just happened?A GitHub Actions workflow is YAML declaring when (on push or PR) and what (jobs of steps) to run on a fresh runner; caching dependencies and Docker layers is the single biggest speedup, and a matrix runs configurations in parallel. Tradeoff: a little YAML to maintain, in exchange for an automated, fast, reproducible pipeline. Next: how that workflow authenticates to the cloud to deploy - without a stored key.

- A job timeline runs cold (long), then cached (short) as the bars shrink
- A matrix toggle splits the run into parallel lanes

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Keyless Deploy with OIDC

### Keyless Deploy with OIDC

Instead of a stored service-account key, the job requests a short-lived OIDC token; the cloud verifies it against an attribute condition (your repo and branch) and grants a credential that expires in about an hour. Nothing is stored to leak or rotate.

To deploy, the CI job must prove to the cloud that it is allowed to. The old way is a **stored service-account JSON key** in your repo secrets - a long-lived credential that never expires on its own, is a nightmare to rotate, and is a disaster if it leaks. The 2026 best practice is **OIDC** (OpenID Connect) **keyless auth**. The job declares `permissions: id-token: write`, and **GitHub mints a short-lived token** whose claims say exactly who is asking (this *repository*, this *branch*). The cloud - **Google Cloud**, via Workload Identity Federation - verifies that token against an **attribute condition** you set (only `your-org/your-repo` on `main` may deploy) and hands back a credential that **expires in about an hour**. The result: **nothing is stored to leak or rotate**, a fork or a wrong branch is denied, and access is scoped to your exact pipeline. One honest nuance: OIDC removes the *cloud deploy* credential, but the eval gate’s LLM-as-judge still needs a provider API key - that stays a normal repo secret (`secrets.ANTHROPIC_API_KEY`) injected only into the eval step. OIDC replaces the standing cloud key, not every third-party key. The block contrasts a stored key with OIDC, keyless.

> 🎟️ **Analogy**
>
> It is a **day-pass badge versus a copied master key**. A stored JSON key is a master key you photocopied and left in a drawer - it opens everything, forever, and if someone finds the copy you are in trouble until you re-key the whole building. OIDC is a day-pass printed at the front desk: it shows your name and department (your repo and branch), it only works if the guard’s list allows you (the attribute condition), and it stops working at the end of the day (it expires in an hour). Nothing to steal from a drawer.

An attacker forks your repo and tries to deploy with your OIDC setup. What stops them?

**📝 `06_oidc_keyless.py`** - *runnable*

In [ ]:
# Two ways for the CI job to authenticate to the cloud in order to deploy.

# (A) A stored service-account JSON key: a long-lived secret sitting in repo settings.
def stored_key(leaked):
    return "stored JSON key: " + ("COMPROMISED - valid until someone manually rotates it" if leaked
                                   else "works, but never expires on its own")

print("Approach A - a stored service-account key (the anti-pattern):")
print("  " + stored_key(leaked=False))
print("  " + stored_key(leaked=True))
print()

# (B) OIDC: GitHub mints a short-lived token with claims (repo, ref); the cloud verifies it
# against an attribute condition (which repo + branch may deploy), then grants a ~1-hour credential.
ALLOW = {"repo": "netsetos/ai-service", "ref": "refs/heads/main"}
def oidc(repo, ref, age_min):
    if repo != ALLOW["repo"] or ref != ALLOW["ref"]:
        return "DENIED by the attribute condition (wrong repo/branch)"
    if age_min > 60:
        return "DENIED - token expired ({} min > 60)".format(age_min)
    return "ALLOWED - short-lived token, no stored secret"

print("Approach B - OIDC keyless (Workload Identity Federation):")
print("  main branch,   5 min old  -> " + oidc("netsetos/ai-service", "refs/heads/main", 5))
print("  a fork,        5 min old  -> " + oidc("attacker/ai-service", "refs/heads/main", 5))
print("  main branch,  75 min old  -> " + oidc("netsetos/ai-service", "refs/heads/main", 75))
print()
print("OIDC: nothing stored to leak or rotate; the token is scoped to your repo/branch and expires in about an hour.")

# Output:
# Approach A - a stored service-account key (the anti-pattern):
#   stored JSON key: works, but never expires on its own
#   stored JSON key: COMPROMISED - valid until someone manually rotates it
#
# Approach B - OIDC keyless (Workload Identity Federation):
#   main branch,   5 min old  -> ALLOWED - short-lived token, no stored secret
#   a fork,        5 min old  -> DENIED by the attribute condition (wrong repo/branch)
#   main branch,  75 min old  -> DENIED - token expired (75 min > 60)
#
# OIDC: nothing stored to leak or rotate; the token is scoped to your repo/branch and expires in about an hour.

- A **stored JSON key** never expires on its own, and if it leaks it stays **valid until someone manually rotates it** - standing risk.
- With **OIDC**, a request from your repo on main with a fresh token is **ALLOWED**.
- A request from a **fork** is **DENIED by the attribute condition** (its repo claim does not match).
- A token older than the lifetime is **DENIED as expired** - nothing stored, scoped to your repo, short-lived.

#### 💡 What just happened

⚡ What just happened?OIDC keyless auth replaces a stored service-account key: the job requests a short-lived token, the cloud verifies its claims against an attribute condition (your repo and branch) via Workload Identity Federation, and grants a credential that expires in about an hour. Tradeoff: a one-time federation setup, in exchange for no stored secret to leak or rotate and access scoped to your exact pipeline. Now the pipeline can deploy - safely - which is the last step.

- A stored key sits in a vault and leaks; vs GitHub minting a short-lived token
- The token is checked against an attribute condition (repo/branch), then expires

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Canary and Rollback

### Canary and Rollback

Deploy the new revision at zero percent traffic, then shift it 10, 50, 100 percent while watching the error rate. A bad canary rolls back to the previous revision in seconds, so a bug never hits everyone at once.

The pipeline passed every gate; now ship it *safely*. Deploying a new revision straight to **100 percent** of traffic means a bug that slipped every gate hits **every user at once**. A **canary** rollout avoids that. On **Google Cloud Run**, you deploy the new revision with **no traffic** (it gets a private URL but zero percent of production), then **shift traffic gradually** - 10 percent, then 50, then 100 - while watching the new revision’s **error rate**. If the canary looks healthy, you promote it to 100 percent. If its error rate **spikes**, you **roll back**: send all traffic to the previous revision, which takes **seconds, not minutes**, because the old revision is still there. The blast radius of a bad deploy shrinks from “everyone” to “a few percent for a few minutes.” The block runs a healthy and a bad canary, keyless.

> 🍲 **Analogy**
>
> It is **tasting one spoonful before serving the whole pot**. You would not ladle a new recipe onto every plate in the restaurant without tasting it first. You taste one spoon (send 10 percent of traffic to the new revision), and if it is good you serve more; if it is off, you set the pot aside and go back to yesterday’s batch (roll back to the previous revision) - and because that batch is still warm on the stove, switching back takes seconds. Only after the spoonful passes does everyone get the new dish.

Your canary is at 50 percent traffic and its error rate spikes to 6 percent, well over your 2-percent bar. What happens?

**📝 `07_canary_rollback.py`** - *runnable*

In [ ]:
# Deploy the new revision at 0% traffic, then shift gradually while watching the error rate.
# Roll back to the previous revision in seconds if the canary misbehaves.
ERR_THRESHOLD = 2.0    # percent; above this, roll back

def rollout(name, canary_errors):
    print(name + ":")
    print("  deploy rev-new --no-traffic --tag=canary    (rev-old 100%, rev-new 0%)")
    for pct, err in zip([10, 50, 100], canary_errors):
        line = "  shift rev-new to {:>3}%   canary error rate {:.1f}%".format(pct, err)
        if err > ERR_THRESHOLD:
            print(line + "   -> SPIKE > {:.0f}%: ROLLBACK to rev-old 100% (seconds)".format(ERR_THRESHOLD))
            print("  final: rev-old 100%, rev-new 0% - bad canary caught before full rollout\n")
            return
        print(line + "   ok")
    print("  final: rev-new 100% - promoted; rev-old kept for an instant rollback\n")

rollout("Scenario A - a healthy canary", [0.4, 0.5, 0.6])
rollout("Scenario B - a bad canary (a regression slipped through)", [0.5, 6.3, 0.0])
print("Canary = ship to a slice, watch the error rate, promote or roll back in seconds - never straight to 100%.")

# Output:
# Scenario A - a healthy canary:
#   deploy rev-new --no-traffic --tag=canary    (rev-old 100%, rev-new 0%)
#   shift rev-new to  10%   canary error rate 0.4%   ok
#   shift rev-new to  50%   canary error rate 0.5%   ok
#   shift rev-new to 100%   canary error rate 0.6%   ok
#   final: rev-new 100% - promoted; rev-old kept for an instant rollback
#
# Scenario B - a bad canary (a regression slipped through):
#   deploy rev-new --no-traffic --tag=canary    (rev-old 100%, rev-new 0%)
#   shift rev-new to  10%   canary error rate 0.5%   ok
#   shift rev-new to  50%   canary error rate 6.3%   -> SPIKE > 2%: ROLLBACK to rev-old 100% (seconds)
#   final: rev-old 100%, rev-new 0% - bad canary caught before full rollout
#
# Canary = ship to a slice, watch the error rate, promote or roll back in seconds - never straight to 100%.

- Both scenarios deploy the new revision **at zero percent** (`--no-traffic --tag=canary`) - a private canary.
- A **healthy** canary’s error rate stays low as traffic shifts 10 → 50 → 100, so it is **promoted**.
- A **bad** canary’s error rate **spikes at 50 percent**, over the bar, so it **rolls back** to the previous revision.
- The rollback takes **seconds** because the old revision is still there - the bug hit a slice of traffic, never everyone.

**📝 `.github/workflows/deploy.yml`** - *illustrative*

```
# .github/workflows/deploy.yml - an illustrative minimal example (GitHub Actions CI/CD for an AI app).
name: deploy
on:
  pull_request:               # on a PR: run the gates (lint, test, eval, scan) - merge-blocking
  push:
    branches: [main]          # on merge to main: also build once + keyless deploy + canary

permissions:
  contents: read
  id-token: write             # required for OIDC keyless auth - no stored cloud key

jobs:
  gates:                      # cheap gates first, so a bad change fails fast
    runs-on: ubuntu-latest
    strategy:
      matrix: {python: ["3.12", "3.13"]}      # lint + tests on both versions in parallel
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v6
        with: {enable-cache: true}            # cache deps - the single biggest speedup
      - run: uv sync --frozen
      - run: uv run ruff check . && uv run pytest tests/           # lint + unit tests

  eval:                       # the EVAL GATE runs ONCE - judge output is interpreter-independent, and it spends real tokens
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v6
        with: {enable-cache: true}
      - run: uv sync --frozen
      - run: uv run promptfoo eval -c promptfooconfig.yaml         # the EVAL GATE (pass-rate) - one billed judge run per PR, not one per Python version
        env: {ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}}  # the LLM-judge key is a STORED secret; OIDC does not cover it

  deploy:
    needs: [gates, eval]                      # only if every gate passed
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: docker/build-push-action@v6     # build ONCE, SHA-tagged, cache layers, then push
        with: {tags: "REGION-docker.pkg.dev/PROJECT/ai/svc:${{ github.sha }}", cache-from: "type=gha", cache-to: "type=gha,mode=max", push: true}
      - run: trivy image --exit-code 1 --severity HIGH,CRITICAL REGION-docker.pkg.dev/PROJECT/ai/svc:${{ github.sha }}    # scan gate - a HIGH/CRITICAL CVE fails the build
      - uses: google-github-actions/auth@v3   # keyless: GitHub OIDC -> Workload Identity Federation
        with: {workload_identity_provider: "projects/.../providers/github", service_account: "deployer@PROJECT.iam.gserviceaccount.com"}
      - uses: google-github-actions/deploy-cloudrun@v3
        with: {service: ai-svc, image: "REGION-docker.pkg.dev/PROJECT/ai/svc:${{ github.sha }}", no_traffic: true, tag: canary}
      - run: gcloud run services update-traffic ai-svc --to-revisions LATEST=10   # canary 10% -> watch -> 50 -> 100

# --- promptfooconfig.yaml (the eval gate config) ---
#   prompts: [file://prompts/assistant.txt]         # prompts as code, versioned in the repo
#   providers: [anthropic:claude-sonnet-4-6]        # pin the model
#   tests: file://tests/eval_cases.yaml             # scored by assertions + an llm-rubric judge
#   defaultTest: {options: {repeat: 3}}             # repeat for non-determinism; gate on the pass rate
# Output: (illustrative minimal example - needs a repo, a WIF pool, and gcloud; run by GitHub, not locally.)
```

- **on: pull_request** runs the gates (lint, tests, the `promptfoo` eval gate) as a merge-blocking check; **on: push to main** also deploys.
- `permissions: id-token: write` enables **OIDC keyless auth**; `setup-uv` with `enable-cache` and the Docker `gha` cache are the caching from Step 5.
- The image is **built once**, SHA-tagged, scanned with `trivy`, then deployed via `google-github-actions/auth` (no stored key) - Step 6.
- The deploy lands a **canary** (`no_traffic`, `tag: canary`) and shifts traffic gradually - Step 7. The `promptfooconfig.yaml` pins the model and repeats each test.

#### 💡 What just happened

⚡ What just happened?A canary deploys the new revision at zero percent traffic, shifts it gradually while watching the error rate, and rolls back to the previous revision in seconds if it spikes - so a bad deploy hits a slice, never everyone. Tradeoff / the whole lesson: a few extra pipeline steps, in exchange for shipping many times a day with a safety net. The full workflow ties every step together: gates on a PR, build once, keyless deploy, canary on merge.

- A traffic dial shifts old/new 100/0 -> 90/10 -> 50/50 -> 0/100
- An error-rate meter; a bad canary trips a seconds-rollback to 100 percent old

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> CI/CD for an AI app is one automated pipeline with two extra gates that normal software does not need. It exists because an LLM app is non-deterministic and can **silently regress quality** while unit tests stay green (Step 1). The pipeline is a **chain of gates** run on every push or PR, cheapest first, a failed gate stopping the line, the image **built once** (Step 2). The AI-specific gates are the **eval gate** - an LLM-as-judge suite gated on a pass-rate threshold with repeats and majority voting (Step 3) - and cheap deterministic **output validators** that also run in production (Step 4). You express it all as a **GitHub Actions workflow** with dependency and layer **caching** (Step 5), authenticate to the cloud with **OIDC keyless auth** instead of a stored key (Step 6), and ship through a **canary** you can roll back in seconds (Step 7). Ask five questions of any AI pipeline: does it **fail fast**, **build once**, **gate on quality**, deploy **without stored keys**, and **roll back in seconds**?

|  | Traditional CI | AI CI/CD |
|---|---|---|
| What a test asserts | an exact return value | answer quality above a bar |
| Determinism | same input, same output | non-deterministic; gate on a pass rate |
| Failure it catches | a crash or wrong value | a silent quality regression |
| Extra gates | lint, unit tests | + eval gate + output validators |
| On every response? | no | validators run in production too |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now ship an AI app through an automated, gated pipeline. Building the **eval set** the gate runs - error analysis and eval-driven development - you meet in Lesson 14.4 and again in Module 14. **Monitoring** the deployed app, which is what the canary’s error-rate meter reads from, comes in Lesson 12.8, and **scaling** it under load comes in Lesson 12.7. The build and scan gates are the Docker work from Lesson 12.5. The primary references are the [GitHub OIDC hardening guide](https://docs.github.com/en/actions/security-for-github-actions/security-hardening-your-deployments/about-security-hardening-with-openid-connect), the [promptfoo GitHub Action](https://github.com/promptfoo/promptfoo-action), [google-github-actions/auth](https://github.com/google-github-actions/auth) for Workload Identity Federation, the [uv GitHub Actions guide](https://docs.astral.sh/uv/guides/integration/github/), and [Cloud Run rollouts and rollbacks](https://docs.cloud.google.com/run/docs/rollouts-rollbacks-traffic-migration).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **CI/CDfor AI Apps**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.6.html` - regenerate this notebook whenever the source HTML is updated.*
